# Sionna 0.19 — Differentiable Ray Tracing Calibration
## Nottingham Urban Area · Ofcom 2018 · 915.95 MHz

**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15  
**Scene:** `scene_with_full_019.xml` — 11 objects, 17 ITU-R P.2040-2 materials  
**Dataset:** Ofcom 2018 drive-test — 1 200 receivers, single TX  
**Reference:** Hoydis et al. 2023 — *Sionna RT: Differentiable Ray Tracing for Radio Propagation Modelling* (arXiv:2311.18558)

---

### Notebook Structure

| Cell | Name | Purpose |
|------|------|---------|
| CELL 0 | Environment Setup | Imports, path setup, TF config |
| CELL 1 | Scene Detection | Auto-detect scene XML, read bounds |
| CELL 2 | Global Configuration | Frequency, TX power, ray parameters |
| CELL 3 | Coordinate Utilities | GPS ↔ local XY, DEM elevation, ray-cast Z |
| CELL 4 | Load Scene | Load Mitsuba scene, configure antennas |
| CELL 4A | Material Properties | Assign ITU-R P.2040-2 EM properties per material |
| CELL 6 | Load Transmitter | GPS → BNG → local XY, ray-cast height AGL |
| CELL 7 | Load Receivers | 1 200 Ofcom measurement points → local XY |
| CELL 8 | Coverage Map | Pre-calibration coverage map (visual check) |
| CELL 8b | Calibration Targets | Load measured RSSI, build calib_receivers list |
| CELL 10 | Trainable Materials | Create `tf.Variable` ε_r, σ, S per material |
| CELL 10b | Scalar Offset Calibration | Baseline: optimise global dB offset (NVLabs ITU-Materials) |
| CELL 11b | Material Calibration | Full diff-RT: optimise ε_r, σ, S via trace_paths + compute_fields |
| CELL 11c | Material Results Plot | Convergence curves + calibrated parameter table |
| CELL 12 | TX Orientation | Optimise TX antenna pointing direction |
| CELL 13 | Post-Calibration Analysis | Final coverage map + per-receiver error statistics |

---

### Key Fixes Applied

| Fix | Problem | Solution |
|-----|---------|---------|
| Scene file | `scene_with_roads_019.xml` (7 objects) loaded | Changed to `scene_with_full_019.xml` (11 objects) |
| GPU OOM | 10M rays × 1 200 RX → 13 GB tensor | Batched: 50 RX × 1M rays per batch |
| RMSE = 149 dB | `eps=1e-30` gave RSSI=−271 dBm (finite, wrong) | Valid mask: `RSSI > −150 dBm` |
| TX double-count | `paths_to_rssi` added `tx_pwr_dbm` twice | Removed — Sionna 0.19 embeds TX power in `paths.a` |
| scaling_factor = −50 dB | TX power counted twice → optimizer compensated | After fix: sf = −1.38 dB ✓ |
| Water ε_r = 30 | mat-water → itu_wet_ground | → itu_water (ε=80, σ=0.010) ITU-R P.527 |
| Vegetation ε_r = 5.31 | mat-vegetation → itu_concrete | → itu_vegetation (ε=1.50, σ=0.0) ITU-R P.833 |


---
## CELL 0 · Environment Setup & Imports

Imports all required libraries and configures TensorFlow GPU memory growth.  
Run this cell first after every kernel restart.


In [ ]:
import os, sys, json, csv, time, warnings, importlib
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from pyproj import Transformer

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

# ── Mitsuba (ray casting) ──────────────────────────────────────────────────────
_HAS_MI = False
try:
    import mitsuba as mi
    try:   mi.set_variant('cuda_ad_mono_polarized')
    except: mi.set_variant('llvm_ad_mono_polarized')
    _HAS_MI = True
    print(f'Mitsuba : {mi.variant()}')
except ImportError:
    print('Mitsuba : NOT available – ray-cast ground height disabled')

# ── rasterio (DEM lookup) ──────────────────────────────────────────────────────
_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
except ImportError:
    print('rasterio: NOT available – DEM elevation disabled')

# ── OFDM helpers (diff-rt NMSE loss) ──────────────────────────────────────────
_HAS_OFDM = False
for _pkg in ('sionna.channel', 'sionna.channel.ofdm'):
    try:
        _m = importlib.import_module(_pkg)
        cir_to_ofdm_channel    = _m.cir_to_ofdm_channel
        subcarrier_frequencies = _m.subcarrier_frequencies
        _HAS_OFDM = True
        print(f'OFDM    : OK  ({_pkg})')
        break
    except (ImportError, AttributeError):
        continue
if not _HAS_OFDM:
    print('OFDM    : NOT found – power-domain fallback will be used')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')
print(f'GPU(s)  : {[g.name for g in tf.config.list_physical_devices("GPU")]}')

# ── Shared helpers ─────────────────────────────────────────────────────────────
def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

---
## CELL 1 · Project & Scene Detection

Auto-detects `scene_with_full_019.xml` from the project folder.  
Reads scene bounds (`WEST`, `EAST`, `SOUTH`, `NORTH`) and scene centre from XML `<default>` tags.  
Sets `XML_OK = True` if the scene file is found — Cell 4 will raise an error if `XML_OK` is False.


In [ ]:
# ── Nottingham DEM scene (EA LiDAR 1m DTM, 915 MHz Ofcom 2018) ───────────────
CITY_NAME    = 'Nottingham'
BASE_DIR     = '/home/georgeskai/sionna_rt/nottingham_ofcom2018_915mhz_dem'
PROJECT_PATH = BASE_DIR
SCENE_XML    = os.path.join(BASE_DIR, 'scene', 'scene_with_full_019.xml')
DEM_TIFF     = os.path.join(BASE_DIR, 'scene', 'dem_wgs84.tif')

# Nottingham DEM scene bbox (must match sionna2_915mhz_dem_simulation CELL 1)
WEST, EAST   = -1.267685, -1.119832
SOUTH, NORTH =  52.943165, 53.003037

XML_OK = os.path.exists(SCENE_XML)
print(f'Project   : {CITY_NAME} — DEM + Roads 915 MHz')
print(f'Base dir  : {BASE_DIR}')
print(f'Scene XML : {SCENE_XML}  {"✓" if XML_OK else "✗ NOT FOUND"}')
print(f'DEM       : {DEM_TIFF}  {"✓" if os.path.exists(DEM_TIFF) else "✗ NOT FOUND"}')
print(f'Bbox      : lon [{WEST}, {EAST}]  lat [{SOUTH}, {NORTH}]')

# Read scene XML to confirm version
if XML_OK:
    try:
        import xml.etree.ElementTree as _ET
        _root = _ET.parse(SCENE_XML).getroot()
        _ver  = _root.get('version', 'unknown')
        _maj  = int(_ver.split('.')[0]) if _ver and _ver[0].isdigit() else 0
        _ok_ver = _ver.startswith('2.') or _ver.startswith('3.')
        print(f'XML version : {_ver}  {"✓ Mitsuba 2.x (Sionna 0.19)" if _ver.startswith("2.") else ("✓ Mitsuba 3.x" if _ver.startswith("3.") else "✗ unknown version")}')
    except Exception as _e:
        print(f'XML parse warning: {_e}')

# Output directory
OUTPUT_DIR = os.path.join(BASE_DIR, 'results', 'diff_rt')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

---
## CELL 2 · Global Configuration

All simulation parameters in one place. Edit here before running the calibration.

| Parameter | Variable | Value | Notes |
|-----------|----------|-------|-------|
| TX conducted power | `TX_CONDUCTED_DBM` | 49.0 dBm | From Ofcom site record |
| RX extra gain | `RX_EXTRA_GAIN_DB` | 0.0 dB | No artificial offsets |
| Max ray depth | `CALIB_DEPTH` | 5 | Reflections per path |
| Rays per batch | `NUM_SAMPLES_PS` | 1 000 000 | Reduced from 10M to prevent GPU OOM |
| Calibration steps | `CALIB_STEPS` | 5 000 | Scalar offset baseline |
| Calibration receivers | `CALIB_N_RX` | 1 200 | All Ofcom measurement points |


In [ ]:
# ── Ofcom site parameters (Nottingham 915 MHz DEM run) ───────────────────────
# Matches sionna2_915mhz_dem_simulation CELL 1 exactly — no offsets.
# Formula: RSSI = TX_CONDUCTED_DBM + 10*log10(sum|a|^2) + RX_EXTRA_GAIN_DB
FREQUENCY_HZ     = 915.95e6     # Ofcom 2018 drive-test frequency
TX_HEIGHT_M      = 17.0         # TX antenna height AGL (m)
TX_CONDUCTED_DBM = 49.0         # dBm (conducted power at antenna port)
TX_GAIN_DBI      = 1.3          # dBi collinear omni — handled by Sionna pattern
EIRP_DBM         = TX_CONDUCTED_DBM  # alias

# RX chain — no corrections applied (matches DEM simulation notebook)
RX_AGL_M         = 1.5
RX_EXTRA_GAIN_DB = 0.0          # no chain gain/loss correction
SYS_GAIN         = 0.0
SITE_CORRECTION_DB = 0.0

# TX GPS position (Ofcom 2018 Nottingham site)
TX_LAT           = 52.9863
TX_LON           = -1.2559

BANDWIDTH_HZ    = 20e6
NOISE_FLOOR     = -109.0    # Ofcom spec: system noise floor (dBm)

# ── Coordinate system ─────────────────────────────────────────────────────────
UTM_EPSG        = 32630   # UTM zone 30N — covers UK/Nottingham

# ── Input / output CSVs ───────────────────────────────────────────────────────
RX_CSV          = os.path.join(BASE_DIR, 'scene', 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(BASE_DIR, 'scene', 'measurements_with_pathloss.csv')
TX_CSV          = os.path.join(BASE_DIR, 'scene', 'transmitter_positions.csv')

# ── OFDM parameters ───────────────────────────────────────────────────────────
NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

# ── Path solver parameters ────────────────────────────────────────────────────
MAX_DEPTH      = 15
NUM_SAMPLES_CM = 1_000_000     # minimal — coverage map is visualisation only
NUM_SAMPLES_PS = 10_000_000   # 10M sufficient for 400m near-field calibration
GRID_SIZE_M    = 5.0

# ── Differentiable RT calibration ─────────────────────────────────────────────
CALIB_STEPS    = 500      # official paper: 10000; 500 is practical for RSSI-only
CALIB_LR       = 5e-3     # Adam learning rate
CALIB_N_RX     = 1200     # all 1200 Ofcom receivers (bad-PL outliers filtered)
CALIB_BATCH    = 8        # NVLabs official batch size
CALIB_NUM_SAMP = 500_000  # num_samples for compute_paths() per step
CALIB_DEPTH    = 5        # NVLabs official depth (3-5)

# ── TX orientation optimization ───────────────────────────────────────────────
ORI_STEPS    = 50
ORI_LR       = 0.01
ORI_NUM_SAMP = 1_000_000

_tx_w    = 10**((EIRP_DBM  - 30) / 10)
_noise_w = 10**((NOISE_FLOOR - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

print('=' * 65)
print('SYSTEM CONFIGURATION — Nottingham 915 MHz DEM (Ofcom 2018)')
print('=' * 65)
print(f'Frequency   : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX EIRP     : {EIRP_DBM} dBm  (pos: {TX_LAT}, {TX_LON}, h={TX_HEIGHT_M}m)')
print(f'SYS_GAIN    : {SYS_GAIN:.1f} dB  (RX extra gain: {RX_EXTRA_GAIN_DB} dB)')
print(f'SNR scale   : {SNR_SCALE:.2e}')
print(f'UTM EPSG    : {UTM_EPSG}')
print(f'Max depth   : {MAX_DEPTH}')
print(f'RX CSV      : {RX_CSV}  {"✓" if os.path.exists(RX_CSV) else "✗  (run CELL 6c in main notebook first)"}')
print(f'Meas CSV    : {MEASUREMENT_CSV}  {"✓" if os.path.exists(MEASUREMENT_CSV) else "✗"}')
print(f'Calib       : {CALIB_STEPS} steps  LR={CALIB_LR}  N={CALIB_N_RX}  batch={CALIB_BATCH}')
print('=' * 65)

---
## CELL 3 · Coordinate Utilities + DEM Elevation

Helper functions for coordinate conversion and height lookup:

| Function | Input | Output |
|----------|-------|--------|
| `gps_to_local(lon, lat)` | WGS84 GPS | Local XY (m) relative to scene origin |
| `local_to_gps(x, y)` | Local XY | WGS84 GPS |
| `get_dem_elevation(x, y)` | Local XY | Absolute height (m ASL) from DTM raster |
| `ray_cast_ground_z(x, y)` | Local XY | Terrain height via Mitsuba ray intersection |

**Coordinate system:** WGS84 GPS → UTM EPSG:32630 (UK zone 30N) → subtract scene origin → Sionna local XY.


In [ ]:
# Derive scene center from bbox
center_lon = (WEST + EAST)   / 2
center_lat = (SOUTH + NORTH) / 2

gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    """GPS (lon, lat) to Sionna local XY (metres from scene origin)."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Sionna local XY to GPS (lon, lat)."""
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

# -- DEM bilinear lookup -----------------------------------------------------
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())
if _is_bng_dem:
    utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    if _is_bng_dem:
        px, py = utm_to_bng.transform(utm_x, utm_y)
    else:
        px, py = utm_to_gps.transform(utm_x, utm_y)  # WGS84 lon/lat
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if not _HAS_MI: return get_dem_elevation(x, y)
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

def _scene_bbox():
    """Return (xmin, xmax, ymin, ymax) in local metres. Works Sionna 0.19 and 2.0."""
    for _attr in ('mi_scene', '_scene'):
        try:
            bb = getattr(scene, _attr).bbox()
            return float(bb.min[0]), float(bb.max[0]), float(bb.min[1]), float(bb.max[1])
        except Exception:
            continue
    # Fallback: compute from lon/lat bbox
    wx, sy = gps_to_utm.transform(WEST,  SOUTH)
    ex, ny = gps_to_utm.transform(EAST,  NORTH)
    return (wx - utm_center_x, ex - utm_center_x,
            sy - utm_center_y, ny - utm_center_y)

print('Coordinate utilities ready.')
print(f'  center: ({center_lon:.4f}, {center_lat:.4f})')

---
## CELL 4 · Load 3-D Scene & Configure Antennas

Loads `scene_with_full_019.xml` — the Sionna 0.19 / Mitsuba scene with 11 geometry objects and 17 ITU-R materials.

**Scene contents:**

| Category | Objects | Materials |
|----------|---------|-----------|
| Buildings | 7 PLY files (brick, concrete, glass, metal, wood variants) | itu_brick, itu_concrete, itu_glass, itu_metal, itu_wood |
| Terrain | terrain.ply (EA LiDAR 1m DTM) | itu_wet_ground |
| Roads | road_itu_asphalt.ply | itu_concrete (asphalt proxy) |
| Water | water.ply (River Trent + Canal) | itu_water (ε=80, σ=0.010) |
| Vegetation | vegetation.ply (parks, gardens) | itu_vegetation (ε=1.50, σ=0.0) |

**Fix applied:** `merge_shapes=True` removed — not supported in Sionna 0.19 `load_scene()`.


In [ ]:
if not XML_OK:
    raise RuntimeError(
        'scene.xml not found. Complete Steps 1-3 in the sionna_web UI:\n'
        '  1. Draw area on map\n'
        '  2. Configure materials\n'
        '  3. Click "Generate 3-D Scene"')

print(f'Loading scene from {SCENE_XML} ...')
# Sionna 2.0: merge_shapes=False keeps named objects for material assignment
try:
    scene = load_scene(SCENE_XML, merge_shapes=False)
except TypeError:
    scene = load_scene(SCENE_XML)   # Sionna 0.19 fallback -- no merge_shapes param
scene.frequency = FREQUENCY_HZ

def _make_array(cfg):
    return PlanarArray(
        num_rows           = cfg.get('num_rows',           1),
        num_cols           = cfg.get('num_cols',           1),
        vertical_spacing   = cfg.get('vertical_spacing',   0.5),
        horizontal_spacing = cfg.get('horizontal_spacing', 0.5),
        pattern            = cfg.get('pattern',            'iso'),
        polarization       = cfg.get('polarization',       'V'),
    )

# Antenna pattern: 'dipole' = half-wave dipole donut (~2.15 dBi, null at zenith/nadir)
# Matches DEM simulation: ANTENNA_PATTERN='donut' -> pattern='dipole'
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='dipole',
                             polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='dipole',
                             polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')
print(f'TX array      : {scene.tx_array}')

# Scene bounding box (uses _scene_bbox() defined in CELL 3)
try:
    _xmin, _xmax, _ymin, _ymax = _scene_bbox()
    print(f'Scene bbox    : X=[{_xmin:.0f}, {_xmax:.0f}]  Y=[{_ymin:.0f}, {_ymax:.0f}]')
except Exception as _be:
    print(f'Scene bbox    : {_be}')


---
## CELL 4A · Assign ITU-R P.2040-2 Material Properties

Assigns electromagnetic properties to each scene material following ITU-R P.2040-2 (buildings),  
ITU-R P.527 (water), and ITU-R P.833 (vegetation) at 915 MHz.

| Material | ε_r | σ (S/m) | Scatter S | Standard | Notes |
|----------|-----|---------|-----------|----------|-------|
| itu_concrete | 5.24 | 0.130 | 0.40 | P.2040-2 | |
| itu_brick | 3.91 | 0.024 | 0.25 | P.2040-2 | |
| itu_glass | 6.27 | 0.012 | 0.08 | P.2040-2 | |
| itu_wood | 1.99 | 0.005 | 0.30 | P.2040-2 | |
| itu_metal | 1.00 | 1×10⁷ | 0.05 | P.2040-2 | Perfect conductor |
| itu_wet_ground | 30.0 | 0.020 | 0.35 | P.2040-2 | Terrain surface |
| **itu_water** | **80.0** | **0.010** | 0.02 | **P.527** | **Fixed: was itu_wet_ground (ε=30)** |
| **itu_vegetation** | **1.50** | **0.000** | 0.10 | **P.833** | **Fixed: was itu_concrete (ε=5.31)** |


In [ ]:
_ITU_DB = {
    'concrete'          : (5.24,  0.130, 0.40, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20),
    'wood'              : (1.99,  0.005, 0.25, 0.30),
    'glass'             : (6.27,  0.012, 0.08, 0.10),
    'metal'             : (1.00,  1e7,   0.05, 0.10),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05),
    'water'             : (81.0,  0.500, 0.02, 0.05),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20),
    'marble'            : (7.07,  0.020, 0.08, 0.10),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20),
    'chipboard'         : (2.58,  0.012, 0.14, 0.20),
    'plywood'           : (2.71,  0.014, 0.15, 0.20),
    'ceiling_board'     : (1.50,  0.006, 0.13, 0.20),
    'floorboard'        : (2.00,  0.010, 0.16, 0.20),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

_SCATTER_CFG = {
    'concrete': (0.15, 0.30), 'brick': (0.10, 0.25), 'wood': (0.10, 0.10),
    'glass': (0.05, 0.01), 'metal': (0.05, 0.001), 'wet_ground': (0.05, 0.10),
    'medium_dry_ground': (0.05, 0.10), 'very_dry_ground': (0.05, 0.10),
    'marble': (0.05, 0.30), 'plasterboard': (0.10, 0.02), 'chipboard': (0.10, 0.02),
    'plywood': (0.10, 0.02), 'ceiling_board': (0.10, 0.02), 'floorboard': (0.10, 0.02),
    'vegetation': (0.75, 1.00), 'asphalt': (0.35, 0.15), 'water': (0.02, 0.10),
}

print('=' * 70)
print('ASSIGNING ITU-R MATERIAL PROPERTIES  (auto-match by name)')
print('=' * 70)
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd = _ITU_DB.get(key, _DEFAULT_MAT)
    sc, th = _SCATTER_CFG.get(key, (0.20, 0.10))
    try: mat.relative_permittivity = eps_r
    except Exception: pass
    try: mat.conductivity = sigma
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, sc); break
            except: pass
    for a_ in ('xpd_coefficient', 'xpd_coeff'):
        if hasattr(mat, a_):
            try: setattr(mat, a_, xpd); break
            except: pass
    try: mat.thickness = th
    except: pass
    print(f'  {mat_name:<32}  matched={key or "DEFAULT":<20}  '
          f'eps={eps_r:.2f}  σ={sigma:.4g}  S={sc:.2f}')
print('Done.')

---
## CELL 6 · Load Transmitter

Reads TX position from `transmitter_positions.csv`.  
Pipeline: GPS (lon, lat) → UTM EPSG:32630 → subtract scene origin → Sionna local XY.  
Height: ray-cast against terrain mesh to get ground Z, then add antenna height AGL.  
TX created with `power_dbm = TX_CONDUCTED_DBM = 49.0` — Sionna 0.19 embeds this into `paths.a`.


In [ ]:
print('=' * 70)
print('CELL 6 – LOAD TRANSMITTER')
print('=' * 70)

for nm in list(scene.transmitters.keys()):
    scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', TX_CONDUCTED_DBM))
    source   = 'CSV'
else:
    # Fallback: use hardcoded Ofcom TX parameters (matches DEM simulation)
    tx_name  = 'tx0'
    tx_lon   = TX_LON
    tx_lat   = TX_LAT
    tx_agl   = TX_HEIGHT_M
    tx_power = TX_CONDUCTED_DBM
    source   = 'hardcoded (TX_LON/TX_LAT/TX_HEIGHT_M)'
    print(f'  TX CSV not found – using hardcoded Ofcom TX parameters')

print(f'[1] Source      : {source}')
print(f'    GPS         : ({tx_lon:.6f}, {tx_lat:.6f})  AGL={tx_agl:.1f} m')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
print(f'[2] Local XY    : ({local_x:.2f}, {local_y:.2f})')

ground_z = ray_cast_ground_z(local_x, local_y)
if ground_z == 0.0:
    try:
        _bbox    = scene.mi_scene.bbox()
        ground_z = float(_bbox.min[2])
    except: pass
abs_z = ground_z + tx_agl
print(f'[3] Ground Z    : {ground_z:.2f} m  +  AGL {tx_agl:.1f} m  →  abs Z={abs_z:.2f} m')

tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)),
                 power_dbm=float(tx_power))
scene.add(tx)
print(f'[4] ✓ Added TX "{tx_name}"  pos=({local_x:.1f}, {local_y:.1f}, {abs_z:.1f})  EIRP={tx_power:.1f} dBm')

---
## CELL 7 · Load Receivers

Reads 1 200 receiver positions from `receiver_locations.csv` and measured RSSI from `measurements_with_pathloss.csv`.  
Same GPS → local XY pipeline as Cell 6.  
All 1 200 receivers are loaded — no distance filter applied.

**Output variables:**
- `receivers` — list of Sionna `Receiver` objects
- `rx_meas_rssi` — measured RSSI (dBm) array, aligned to receivers list


In [ ]:
print('=' * 70)
print('CELL 7 – LOAD RECEIVERS')
print('=' * 70)

for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV).head(1200)  # cap at first 1200 RX
    print(f'[1] Loaded {len(df_rx)} receivers (capped at 1200) from {RX_CSV}')
    print('[2] Converting GPS → local XY + ray-cast ground Z ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon  = float(row['lon'])
        lat  = float(row['lat'])
        agl  = float(row.get('height', 1.5))
        x, y, _ = gps_to_local(lon, lat)
        gz   = ray_cast_ground_z(x, y)
        z    = gz + agl
        nm   = str(row.get('name', f'RX_{i+1:04d}'))
        rx   = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx)
        receivers.append(rx)
    print(f'    Done in {time.time()-t0:.2f} s')
else:
    print('  RX CSV not found – using project.json receivers')
    for rx_cfg in _ant.get('receivers', [{'name':'rx0','position':[100,0,1.5]}]):
        pos = rx_cfg.get('position', [100, 0, 1.5])
        nm  = rx_cfg.get('name', f'rx{len(receivers)}')
        rx  = Receiver(name=nm, position=(float(pos[0]), float(pos[1]), float(pos[2])))
        scene.add(rx)
        receivers.append(rx)

print(f'[3] {len(receivers)} receivers placed')
print('[4] First 5 receivers:')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f}, {y:8.1f})  Z={z:.2f}  GPS=({lon:.5f}, {lat:.5f})')

try:
    _bbox = scene.mi_scene.bbox()
    ok_x  = all(float(_bbox.min[0]) <= _safe(r.position[0]) <= float(_bbox.max[0]) for r in receivers)
    ok_y  = all(float(_bbox.min[1]) <= _safe(r.position[1]) <= float(_bbox.max[1]) for r in receivers)
    print(f'[5] All inside scene bbox: X={ok_x}  Y={ok_y}')
except: pass

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

# Load Ofcom measurements for calibration
df_meas = None
rssi_measured_all = None
if os.path.exists(MEASUREMENT_CSV):
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    _rssi_col = next((c for c in df_meas.columns
                      if any(k in c.lower() for k in ['rssi','measurement','dbm','signal'])), None)
    if _rssi_col:
        rssi_measured_all = df_meas[_rssi_col].values.astype(np.float32)
        print(f'Ofcom RSSI loaded : {len(rssi_measured_all)} samples  '
              f'range {rssi_measured_all.min():.1f}\u2013{rssi_measured_all.max():.1f} dBm')
    else:
        print(f'WARNING: no RSSI column in {MEASUREMENT_CSV}')
else:
    print(f'WARNING: {MEASUREMENT_CSV} not found \u2014 run CELL 6c in main notebook first')


---
## CELL 8 · Pre-Calibration Coverage Map

Generates a coverage map using ITU-R default material properties — a visual sanity check  
before calibration begins. Shows predicted RSSI across the scene at 20m resolution.  
This cell is optional; skip if runtime is a concern.


In [ ]:
print('Pre-calibration coverage map (ITU material defaults) ...')

try:
    _bbox = scene.mi_scene.bbox()
    cx = (float(_bbox.min[0]) + float(_bbox.max[0])) / 2
    cy = (float(_bbox.min[1]) + float(_bbox.max[1])) / 2
except Exception:
    cx = cy = 0.0

ground_z_centre = ray_cast_ground_z(cx, cy)
if ground_z_centre == 0.0:
    ground_z_centre = get_dem_elevation(cx, cy)
cm_height = ground_z_centre + 1.5
print(f'  Coverage map plane Z = {ground_z_centre:.2f} + 1.5 = {cm_height:.2f} m')

try:
    cm_pre = scene.coverage_map(
        cm_cell_size        = [GRID_SIZE_M, GRID_SIZE_M],
        max_depth           = MAX_DEPTH,
        num_samples         = NUM_SAMPLES_CM,
        los                 = True,
        specular_reflection = True,
        diffuse_reflection  = True,
        refraction          = True,
        diffraction         = True,
    )
except TypeError:
    # Sionna 0.19 uses different parameter names
    cm_pre = scene.coverage_map(
        cm_cell_size = [GRID_SIZE_M, GRID_SIZE_M],
        max_depth    = MAX_DEPTH,
        num_samples  = NUM_SAMPLES_CM,
    )
cm_pre_np = _cm_to_numpy(cm_pre)

pg_db = 10 * np.log10(cm_pre_np[0] + 1e-30)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pg_db, origin='lower', cmap='jet',
               vmin=np.nanpercentile(pg_db, 5), vmax=np.nanpercentile(pg_db, 99))
plt.colorbar(im, ax=ax, label='Path Gain (dB)')
ax.set_title('Pre-Calibration Coverage Map – ITU Material Defaults')
ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_pre_calibration.png'), dpi=150)
plt.show()

# ── Interpolate to RX positions via KDTree ────────────────────────────────────
H, W = pg_db.shape
try:
    _bbox = scene.mi_scene.bbox()
    _gx_min, _gx_max = float(_bbox.min[0]), float(_bbox.max[0])
    _gy_min, _gy_max = float(_bbox.min[1]), float(_bbox.max[1])
except Exception:
    _gx_min = _gy_min = -500.0; _gx_max = _gy_max = 500.0

x_centers = np.linspace(_gx_min, _gx_max, W)
y_centers  = np.linspace(_gy_min, _gy_max, H)
XX, YY = np.meshgrid(x_centers, y_centers)
tree   = KDTree(np.column_stack([XX.ravel(), YY.ravel()]))
rx_coords   = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
_, _indices = tree.query(rx_coords)
pg_at_rx_pre = pg_db.ravel()[_indices]

print(f'Pre-calibration path-gain at RX:  '
      f'mean={np.mean(pg_at_rx_pre):.1f} dB  '
      f'min={np.min(pg_at_rx_pre):.1f} dB  max={np.max(pg_at_rx_pre):.1f} dB')

---
## CELL 8b · Calibration Targets

Builds the calibration dataset:
- `calib_receivers` — list of `Receiver` objects to use in calibration
- `calib_rssi_meas` — matching measured RSSI (dBm) tensor

All 1 200 receivers are included (distance filter removed).  
Measured RSSI loaded from `measurements_with_pathloss.csv` column `RSSI_dBm`.


In [ ]:
# ====================================================================
# CELL 8b — CALIBRATION TARGET
# ====================================================================
# If Ofcom measurements are available: use measured RSSI dBm as target
# (power-domain calibration — correct approach for drive-test CSV data)
# Fallback: self-supervised NMSE using compute_paths() (diff-rt demo mode)
# ====================================================================

CALIB_MODE = 'ofcom'   # 'ofcom' = use measured RSSI  |  'self' = self-supervised NMSE

if CALIB_MODE == 'ofcom' and rssi_measured_all is not None:
    # ── Stratified sample of CALIB_N_RX receivers ──────────────────────────
    import math as _math
    _rx_names = [rx.name for rx in receivers]
    _rx_rssi  = {r: rssi_measured_all[i] for i, r in enumerate(_rx_names)
                 if i < len(rssi_measured_all) and np.isfinite(rssi_measured_all[i])}

    # Compute distances for stratified sampling
    _tx_x, _tx_y = gps_to_local(TX_LON, TX_LAT)[:2]
    _dists = {rx.name: float(np.sqrt(
        (_safe(rx.position[0]) - _tx_x)**2 +
        (_safe(rx.position[1]) - _tx_y)**2)) / 1000.0
        for rx in receivers}

    # Stratified: equal-count bins across distance range
    _valid_rx = [rx for rx in receivers if rx.name in _rx_rssi]
    _valid_rx.sort(key=lambda r: _dists[r.name])
    # No distance filter — use all receivers across full scene extent
    print(f'  Distance filter : disabled — using all {len(_valid_rx)} RX')
    # ── Filter out bad path-loss measurements (outliers) ─────────────────────
    _pl_vals = np.array([49.0 - _rx_rssi[rx.name] for rx in _valid_rx
                         if rx.name in _rx_rssi])
    _pl_med  = float(np.median(_pl_vals))
    _pl_std  = float(np.std(_pl_vals))
    _pl_lo   = _pl_med - 3.0 * _pl_std   # lower bound
    _pl_hi   = _pl_med + 3.0 * _pl_std   # upper bound
    _valid_rx = [rx for rx in _valid_rx
                 if _pl_lo <= (49.0 - _rx_rssi[rx.name]) <= _pl_hi]
    print(f'  Bad-PL filter   : kept {len(_valid_rx)} / {len([r for r in receivers if r.name in _rx_rssi])} '
          f'(|PL - median| <= 3σ,  range [{_pl_lo:.1f}, {_pl_hi:.1f}] dB)')


    _n_bins  = max(1, CALIB_N_RX // 20)
    _bin_sz  = max(1, len(_valid_rx) // _n_bins)
    _sel_idx = []
    for _b in range(_n_bins):
        _bin = _valid_rx[_b*_bin_sz : (_b+1)*_bin_sz]
        _k   = max(1, CALIB_N_RX // _n_bins)
        _step = max(1, len(_bin) // _k)
        _sel_idx += [receivers.index(r) for r in _bin[::_step]][:_k]
    _sel_idx = sorted(set(_sel_idx))[:CALIB_N_RX]

    calib_receivers  = [receivers[i] for i in _sel_idx]
    calib_rssi_meas  = tf.constant(
        [_rx_rssi[rx.name] for rx in calib_receivers], dtype=tf.float32)

    h_ref_tf = None  # not used in 'ofcom' mode

    print(f'Calibration mode  : Ofcom RSSI  ({len(calib_receivers)} receivers)')
    print(f'RSSI range        : {float(calib_rssi_meas.numpy().min()):.1f} – '
          f'{float(calib_rssi_meas.numpy().max()):.1f} dBm')
    _d_sel = [_dists[rx.name] for rx in calib_receivers]
    print(f'Distance range    : {min(_d_sel):.2f} – {max(_d_sel):.2f} km')

else:
    # ── Self-supervised fallback ────────────────────────────────────────────
    CALIB_MODE = 'self'
    print('Calibration mode  : self-supervised NMSE (no Ofcom data)')
    print(f'Computing reference channel  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

    def compute_h_freq(sc, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH):
        paths = sc.compute_paths(
            max_depth=depth, num_samples=num_samp,
            los=True, reflection=True, scattering=True, diffraction=True)
        if _HAS_OFDM:
            try:
                a, tau = paths.cir()
                h = cir_to_ofdm_channel(FREQUENCIES, a, tau, normalize=False)
                return tf.squeeze(h)
            except Exception as _e:
                print(f'  cir() fallback: {_e}')
        a_t = paths.a
        if isinstance(a_t, tuple): a_t = tf.complex(a_t[0], a_t[1])
        if a_t.shape[0] == 1: a_t = a_t[0]
        return tf.cast(tf.reduce_sum(tf.abs(a_t)**2,
                        axis=list(range(1, len(a_t.shape)))), tf.float32)

    h_ref    = compute_h_freq(scene, num_samp=NUM_SAMPLES_PS, depth=MAX_DEPTH)
    h_ref_np = _to_numpy(h_ref)
    h_ref_tf = tf.constant(h_ref_np,
                            dtype=tf.complex64 if np.iscomplexobj(h_ref_np) else tf.float32)
    calib_receivers = list(receivers)
    calib_rssi_meas = None
    print(f'Reference shape   : {h_ref_np.shape}  dtype={h_ref_np.dtype}')

print(f'\nCalibration receivers : {len(calib_receivers)}')
print(f'Mode                  : {CALIB_MODE}')


---
## CELL 10 · Differentiable RT — Setup

Prepares the differentiable calibration pipeline. Contains three sub-cells:

| Sub-cell | Contents | Must run? |
|----------|----------|-----------|
| **CELL 10 (this code cell)** | Creates `trainable_mats` with `tf.Variable` ε_r, σ, S — legacy NVLabs pattern | Yes — defines `_match_itu()`, `_ITU_DB`, `_DEFAULT_MAT` |
| **CELL 10 Loss Functions** | Defines `paths_to_rssi()`, `smape_power_loss()`, `mse_dbm_loss()` | Yes — used by both Cell 10b and Cell 11b |
| **CELL 10b Scalar Offset** | Baseline calibration — optimises a single global dB offset | Optional — run for RMSE=5.72 dB baseline |

**RSSI formula (Sionna 0.19):**
```
RSSI_dBm = 10·log₁₀(Σᵢ|aᵢ|²) + 30 + sys_gain_dB
```
TX power is NOT added — Sionna 0.19 embeds `power_dbm` into `paths.a`.  
Reference: Hoydis et al. 2023, §III.

**Loss function — SMAPE on linear power (NVLabs standard):**
```
L = mean( |P_sim − P_meas| / (P_sim + P_meas + ε) )
```


In [ ]:
orig_params    = {}
original_mats  = {}
trainable_mats = {}
_train_suffix  = '_train'

print('Creating trainable RadioMaterial objects ...')
for mat_name, mat in list(scene.radio_materials.items()):
    if mat_name.endswith(_train_suffix): continue

    # Train all ITU materials that appear in the scene XML
    # (is_used unreliable in Sionna 0.19 — train any material matching ITU pattern)
    _itu_prefixes = ('itu_', 'mat-itu_')
    _used = any(mat_name.startswith(p) for p in _itu_prefixes)
    if not _used:
        # Also check if any scene object uses this material
        _used = any(
            getattr(getattr(obj, 'radio_material', None), 'name', '') == mat_name
            for obj in scene.objects.values())
    if not _used: continue

    key  = _match_itu(mat_name)
    _itu = _ITU_DB.get(key, _DEFAULT_MAT)
    eps0 = _itu[0]; sig0 = _itu[1]; S0 = _itu[2]
    try:
        v = mat.relative_permittivity
        eps0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    try:
        v = mat.conductivity
        sig0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    for a_ in ('scattering_coefficient','scattering_coeff'):
        if hasattr(mat, a_):
            try: S0 = float(getattr(mat,a_).numpy() if hasattr(getattr(mat,a_),'numpy') else getattr(mat,a_)); break
            except: pass

    orig_params[mat_name] = {'eps_r': eps0, 'sigma': sig0, 'S': S0}

    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    # Log-parameterise conductivity for numerical stability (spans 9 orders of magnitude)
    _log_sig0 = float(np.log(max(sig0, 1e-6)))
    kw = dict(
        relative_permittivity = tf.Variable(eps0,     dtype=tf.float32, name=f'{sn}_eps'),
        conductivity          = tf.Variable(_log_sig0, dtype=tf.float32, name=f'{sn}_log_sig'),
        # NOTE: conductivity variable stores LOG(sigma) — exponentiated when assigned to mat
    )
    try:
        new_mat = RadioMaterial(mat_name+_train_suffix,
                                scattering_coefficient=tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S'),
                                **kw)
    except TypeError:
        new_mat = RadioMaterial(mat_name+_train_suffix, **kw)
        try: new_mat.scattering_coefficient = tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S')
        except: pass

    # Assign exp(log_sigma) as actual conductivity
    try:
        new_mat.conductivity = tf.exp(kw['conductivity'])
    except Exception:
        pass

    scene.add(new_mat)
    original_mats[mat_name]  = mat
    trainable_mats[mat_name] = new_mat
    print(f'  {mat_name:<30} → {mat_name+_train_suffix}')
    print(f'    eps_r={eps0:.3f}  log_sig={_log_sig0:.4g}  S={S0:.2f}')

print()
n_redir = 0
for obj_name, obj in scene.objects.items():
    rm = getattr(obj, 'radio_material', None)
    if rm is None: continue
    orig_name = rm.name if hasattr(rm,'name') else str(rm)
    if orig_name in trainable_mats:
        try:
            obj.radio_material = orig_name + _train_suffix
            n_redir += 1
        except Exception as e:
            print(f'  WARNING [{obj_name}]: {e}')

print(f'Redirected {n_redir} scene objects to trainable materials.')
print(f'Trainable materials: {list(trainable_mats.keys())}')


In [ ]:
# ====================================================================
# LOSS FUNCTIONS — matching CALIB_MODE
# ====================================================================

def smape_power_loss(rssi_sim_dbm, rssi_meas_dbm):
    """
    SMAPE on linear power — official diff-rt-calibration loss (Hoydis et al. 2023).
    More robust than MSE on dBm: scale-invariant, symmetric.
    """
    P_sim  = tf.pow(10.0, (rssi_sim_dbm  - 30.0) / 10.0)   # dBm → Watts
    P_meas = tf.pow(10.0, (rssi_meas_dbm - 30.0) / 10.0)
    P_sim  = tf.cast(P_sim,  tf.float32)
    P_meas = tf.cast(P_meas, tf.float32)
    return tf.reduce_mean(
        tf.abs(P_sim - P_meas) / (P_sim + P_meas + 1e-30))

def mse_dbm_loss(rssi_sim_dbm, rssi_meas_dbm):
    """MSE on dBm — simpler alternative, biased toward strong signals."""
    err = tf.cast(rssi_sim_dbm, tf.float32) - tf.cast(rssi_meas_dbm, tf.float32)
    return tf.reduce_mean(err ** 2)

def nmse_loss(h_hat, h_ref):
    """NMSE — used in self-supervised mode only."""
    h_hat = tf.cast(h_hat, h_ref.dtype)
    err   = tf.reduce_mean(tf.abs(h_hat - h_ref)**2)
    ref   = tf.reduce_mean(tf.abs(h_ref)**2) + 1e-30
    return err / ref

def paths_to_rssi(paths, tx_pwr_dbm, sys_gain_db, eps=1e-30):
    """Extract total received power from paths → RSSI dBm per receiver.

    Sionna 0.19 with power_dbm set on Transmitter embeds TX power into
    paths.a. So sum|a|² = P_rx in Watts already — do NOT add tx_pwr_dbm.
    Formula: RSSI_dBm = 10*log10(P_rx_watts) + 30  (W → dBm)
    Reference: NVLabs diff-rt-calibration, Hoydis et al. 2023.
    """
    a_t = paths.a
    if isinstance(a_t, tuple):
        a_t = tf.complex(a_t[0], a_t[1])
    a_t = tf.cast(a_t, tf.complex64)
    pwr_all = tf.abs(a_t)**2
    n_rx = tf.shape(pwr_all)[0]
    pwr = tf.reduce_sum(tf.reshape(pwr_all, [n_rx, -1]), axis=1)
    pwr = tf.cast(pwr, tf.float32)
    pwr = tf.squeeze(pwr)
    # P_rx_watts → dBm (+ sys_gain, no tx_pwr_dbm — already in paths.a)
    rssi_dbm = 10.0 * tf.experimental.numpy.log10(pwr + eps) + 30.0 + sys_gain_db
    return rssi_dbm

def check_mat(mat):
    """Clamp material properties to physical range."""
    try:
        v = mat.relative_permittivity
        if hasattr(v, 'assign'):
            v.assign(tf.clip_by_value(v, 1.0, 50.0))
    except Exception: pass
    try:
        v = mat.conductivity
        if hasattr(v, 'assign'):
            # If log-parameterised: keep log_sigma in reasonable range
            v.assign(tf.clip_by_value(v, tf.math.log(1e-6), tf.math.log(1e7)))
    except Exception: pass
    for a_ in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, a_):
            try:
                v = getattr(mat, a_)
                if hasattr(v, 'assign'):
                    v.assign(tf.clip_by_value(v, 1e-3, 1.0 - 1e-3))
                break
            except Exception: pass

print(f'Loss functions ready — active mode: {CALIB_MODE}')
print(f'  Ofcom mode : SMAPE on linear power (Hoydis et al. 2023)')
print(f'  Self mode  : NMSE on OFDM channel')


---
## CELL 10b · Scalar Offset Calibration — Baseline

**Purpose:** Establish a baseline RMSE before material parameter tuning.  
Follows the NVLabs *"ITU Materials"* baseline from Hoydis et al. 2023.

**Method:**
1. Pre-trace all paths once with `compute_paths()` — fast, no gradient needed
2. Cache the RSSI values from `paths.a` (fixed, no ray tracing in the loop)
3. Optimise a single scalar `scaling_factor_db` with Adam to minimise SMAPE

**Why scalar offset?**  
A global dB shift aligns the overall power scale. It corrects systematic errors  
(e.g. antenna gain uncertainty) but cannot fix spatially varying multipath errors.  
This is the ceiling for a single-parameter model.

| Parameter | Value |
|-----------|-------|
| Variable | `scaling_factor_db` — global dB shift |
| Optimiser | Adam, LR = 0.5 |
| Steps | `CALIB_STEPS` = 5 000 |
| Loss | SMAPE on linear power |
| Pre-trace | `compute_paths()`, 50 RX/batch, 1M rays/batch |
| Valid mask | RSSI > −150 dBm (excludes zero-power receivers) |

**Expected result:** `scaling_factor_db` ≈ −1.4 dB, RMSE ≈ 5.7 dB


In [ ]:
# ====================================================================
# CELL 10b — NVLabs-style calibration: pre-trace once, then optimise
# ====================================================================
# Step 1: compute_paths() runs ONCE for all calib receivers (offline)
# Step 2: training loop optimises scaling_factor_db — no ray tracing
#          inside the tape → gradient always non-zero (matches NVLabs
#          "ITU Materials" baseline from Hoydis et al. 2023)
# ====================================================================
import random, time

# ── Step 1: pre-trace in batches (avoids GPU OOM) ───────────────────────────
_BATCH_SIZE  = 50       # receivers per batch — reduce if still OOM
_NUM_SAMPLES = 1_000_000  # rays per batch — lower than full scene pre-trace
print(f'Pre-tracing paths ({len(calib_receivers)} receivers, batch={_BATCH_SIZE}, samples={_NUM_SAMPLES:,}) ...')
print(f'  depth={CALIB_DEPTH}  batches={int(np.ceil(len(calib_receivers)/_BATCH_SIZE))}')

_ps_cfg = dict(
    max_depth   = CALIB_DEPTH,
    num_samples = _NUM_SAMPLES,
    los         = True,
    diffraction = True,
)

_rssi_batches = []
for _b0 in range(0, len(calib_receivers), _BATCH_SIZE):
    _batch = calib_receivers[_b0 : _b0 + _BATCH_SIZE]
    for nm in list(scene.receivers.keys()):
        scene.remove(nm)
    for rx in _batch:
        scene.add(rx)
    try:
        _paths_b = scene.compute_paths(reflection=True, scattering=False, **_ps_cfg)
    except TypeError:
        _paths_b = scene.compute_paths(specular_reflection=True,
                                       diffuse_reflection=False,
                                       refraction=True, **_ps_cfg)
    _rssi_b = paths_to_rssi(_paths_b, TX_CONDUCTED_DBM, RX_EXTRA_GAIN_DB)
    _rssi_b = tf.reshape(_rssi_b, [-1])
    _rssi_batches.append(_rssi_b.numpy())
    if (_b0 // _BATCH_SIZE) % 5 == 0:
        print(f'  batch {_b0//_BATCH_SIZE+1}/{int(np.ceil(len(calib_receivers)/_BATCH_SIZE))}  '
              f'solved={int(np.sum(np.isfinite(_rssi_b.numpy())))}/{len(_batch)}')

rssi_sim_cached = tf.constant(np.concatenate(_rssi_batches), dtype=tf.float32)
n_solved = int(tf.reduce_sum(tf.cast(tf.math.is_finite(rssi_sim_cached), tf.int32)).numpy())
print(f'Paths solved : {n_solved}/{len(calib_receivers)} receivers')
print(f'RSSI_sim     : {float(tf.reduce_min(rssi_sim_cached).numpy()):.1f} – {float(tf.reduce_max(rssi_sim_cached).numpy()):.1f} dBm')

# ── Align calib_rssi_meas to rssi_sim_cached length ──────────────────────────
_n_common  = min(len(rssi_sim_cached), len(calib_rssi_meas))
_meas_trim = calib_rssi_meas[:_n_common]
_sim_trim  = rssi_sim_cached[:_n_common]

# Keep only receivers with real paths (RSSI > -150 dBm)
# eps=1e-30 in paths_to_rssi gives RSSI=-251 dBm for zero-power receivers
# — these are finite but wrong; exclude them with a threshold
_valid_mask       = tf.math.is_finite(_sim_trim) & (_sim_trim > -150.0)
rssi_sim_valid    = tf.boolean_mask(_sim_trim,  _valid_mask)
rssi_meas_valid   = tf.boolean_mask(_meas_trim, _valid_mask)
print(f'Valid pairs  : {int(rssi_sim_valid.shape[0])} (RSSI > -150 dBm threshold)')
if int(rssi_sim_valid.shape[0]) > 0:
    print(f'RSSI_sim     : {float(tf.reduce_min(rssi_sim_valid).numpy()):.1f} – {float(tf.reduce_max(rssi_sim_valid).numpy()):.1f} dBm')
    print(f'RSSI_meas    : {float(tf.reduce_min(rssi_meas_valid).numpy()):.1f} – {float(tf.reduce_max(rssi_meas_valid).numpy()):.1f} dBm')

# ── Step 2: optimise scaling_factor_db ───────────────────────────────────────
# scaling_factor_db: global dB shift that aligns sim power to measurements
# Gradient: d(SMAPE)/d(sf) is always non-zero → zero_grads = 0/1
scaling_factor_db = tf.Variable(0.0, dtype=tf.float32, trainable=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.5)   # large LR ok for scalar

# ── Initial RMSE before training ─────────────────────────────────────────────
_rmse_init = float(tf.sqrt(tf.reduce_mean(((rssi_sim_valid - rssi_meas_valid)**2))).numpy())
_mae_init  = float(tf.reduce_mean(tf.abs(rssi_sim_valid - rssi_meas_valid)).numpy())
print(f'Before calibration : RMSE={_rmse_init:.2f} dB   MAE={_mae_init:.2f} dB  (N={int(rssi_sim_valid.shape[0])})')

history = {'step': [], 'loss': [], 'sf_db': [], 'rmse': []}
t0 = time.time()

print(f'\nTraining scaling_factor  ({CALIB_STEPS} steps, LR=0.5)')
print('-' * 55)

for step in range(CALIB_STEPS):
    with tf.GradientTape() as tape:
        rssi_scaled = rssi_sim_valid + scaling_factor_db
        loss = smape_power_loss(rssi_scaled, rssi_meas_valid)

    grads = tape.gradient(loss, [scaling_factor_db])
    optimizer.apply_gradients(zip(grads, [scaling_factor_db]))

    lv  = float(loss.numpy()) * 100
    sfv = float(scaling_factor_db.numpy())
    history['step'].append(step)
    history['loss'].append(lv)
    history['sf_db'].append(sfv)

    if step % 50 == 0 or step == CALIB_STEPS - 1:
        n_z = sum(1 for g in grads
                  if g is None or float(tf.reduce_sum(tf.abs(g))) == 0)
        _rmse_v = float(tf.sqrt(tf.reduce_mean(((rssi_sim_valid + scaling_factor_db - rssi_meas_valid)**2))).numpy())
        print(f'  step {step:4d}  RMSE={_rmse_v:5.2f} dB  SMAPE×100={lv:+7.2f}  sf={sfv:+.2f} dB  t={time.time()-t0:.0f}s')

print('-' * 55)
print(f'Done in {time.time()-t0:.1f}s')
_rssi_cal  = rssi_sim_valid + float(scaling_factor_db.numpy())
_rmse_final = float(tf.sqrt(tf.reduce_mean(((_rssi_cal - rssi_meas_valid)**2))).numpy())
_mae_final  = float(tf.reduce_mean(tf.abs(_rssi_cal - rssi_meas_valid)).numpy())
print(f'Calibrated scaling_factor = {float(scaling_factor_db.numpy()):+.4f} dB')
print(f'After  calibration : RMSE={_rmse_final:.2f} dB   MAE={_mae_final:.2f} dB')
print(f'RMSE improvement   : {_rmse_init - _rmse_final:+.2f} dB')


---
## CELL 11b · Material Parameter Calibration — NVLabs Differentiable RT

**Purpose:** Reduce RMSE below the scalar-offset baseline by optimising physical material properties.

**Method (Hoydis et al. 2023, §IV):**
1. Pre-trace geometry once with `trace_paths()` — geometry is fixed, only EM changes
2. Inside `tf.GradientTape`: update materials → `compute_fields()` → RSSI → SMAPE loss
3. Adam applies gradients to ε_r, log(σ), S for each ITU material

**Key design decisions:**

| Decision | Reason |
|----------|--------|
| `trace_paths()` outside tape | Geometry is non-differentiable — only EM fields depend on materials |
| `compute_fields()` inside tape | This is differentiable w.r.t. material params in Sionna 0.19 |
| log(σ) parameterisation | σ spans 9 orders of magnitude (10⁻⁶ to 10⁷ S/m) — log-space is numerically stable |
| Receivers restored per batch | `compute_fields()` requires same receivers as `trace_paths()` — critical fix |
| Physical bounds | ε_r ∈ [1,100], σ ∈ [10⁻⁶,10⁷], S ∈ [0,1] enforced via `tf.clip_by_value` |

**Variables optimised:** ε_r, σ, S for each of: itu_concrete, itu_brick, itu_glass, itu_wood, itu_wet_ground, itu_water, itu_vegetation

| Parameter | Value |
|-----------|-------|
| Optimiser | Adam, LR = 0.01 |
| Steps | 200 |
| Loss | SMAPE on linear power |
| Pre-trace | `trace_paths()`, 50 RX/batch, 500k rays/batch |
| Output | Calibrated ε_r, σ, S table + RMSE before/after |

**After this cell:** Copy calibrated values into `sionna2_915mhz_dem_simulation.ipynb` Cell 4A  
to re-run the full DEM simulation with optimised materials.


In [ ]:
# ====================================================================
# CELL 11b — Material Parameter Calibration (NVLabs differentiable RT)
# Sionna 0.19 API: trace_paths() returns a tuple of 8 path objects;
# compute_fields(*traced_tuple) unpacks them correctly.
# ====================================================================
import time
import numpy as np
import tensorflow as tf

MAT_STEPS   = 200
MAT_LR      = 1e-2
MAT_BATCH   = 20
MAT_SAMPLES = 200_000
MAT_DEPTH   = CALIB_DEPTH

print('=' * 70)
print('CELL 11b — Material Parameter Calibration')
print('=' * 70)

# ── 1. Trainable variables per material ─────────────────────────────────
_MAT_BOUNDS = {
    'concrete'   : (2.0,  10.0,  0.01,  1.0,   0.0, 0.8),
    'brick'      : (2.0,   8.0,  0.001, 0.5,   0.0, 0.8),
    'glass'      : (3.0,  10.0,  0.0,   0.1,   0.0, 0.5),
    'wood'       : (1.5,   5.0,  0.0,   0.1,   0.0, 0.8),
    'metal'      : (1.0,   1.0,  1e4,   1e8,   0.0, 0.3),
    'wet_ground' : (5.0,  40.0,  0.001, 0.1,   0.0, 0.5),
    'water'      : (60.0, 90.0,  0.001, 0.05,  0.0, 0.3),
    'vegetation' : (1.2,   5.0,  0.0,   0.1,   0.0, 0.9),
}
mat11_vars = {}
mat11_init = {}
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    if key not in _MAT_BOUNDS:
        continue
    try:    eps0 = float(np.real(mat.relative_permittivity.numpy()))
    except: eps0 = _ITU_DB.get(key, _DEFAULT_MAT)[0]
    try:    sig0 = float(np.real(mat.conductivity.numpy()))
    except: sig0 = _ITU_DB.get(key, _DEFAULT_MAT)[1]
    sig0 = max(sig0, 1e-6)
    s0 = 0.0
    for _a in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, _a):
            try: s0 = float(getattr(mat, _a).numpy()); break
            except: pass
    mat11_init[mat_name] = {'eps': eps0, 'sig': sig0, 's': s0}
    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    mat11_vars[mat_name] = {
        'eps'    : tf.Variable(eps0,                dtype=tf.float32, name=f'{sn}_eps'),
        'log_sig': tf.Variable(float(np.log(sig0)), dtype=tf.float32, name=f'{sn}_lsig'),
        's'      : tf.Variable(s0,                  dtype=tf.float32, name=f'{sn}_s'),
    }
print(f'Materials to calibrate : {len(mat11_vars)}')
for mn in mat11_vars:
    iv = mat11_init[mn]
    print(f'  {mn:<30}  eps_r={iv["eps"]:.3f}  sigma={iv["sig"]:.5f}  S={iv["s"]:.3f}')

# ── 2. Pre-trace geometry ─────────────────────────────────────────────────
# trace_paths() returns a tuple of 8 path objects:
#   (spec_paths, diff_paths, scat_paths, ris_paths,
#    spec_paths_tmp, diff_paths_tmp, scat_paths_tmp, ris_paths_tmp)
# Store the full tuple — unpack with * when calling compute_fields()
print(f'\nPre-tracing ({len(calib_receivers)} rx, batch={MAT_BATCH}, samples={MAT_SAMPLES:,}) ...')
_tr_cfg = dict(max_depth=MAT_DEPTH, num_samples=MAT_SAMPLES,
               los=True, reflection=True, diffraction=True, scattering=True)
_traced_list = []   # (paths_tuple, [rx_objects])
_meas_list   = []

for _b0 in range(0, len(calib_receivers), MAT_BATCH):
    _brx = calib_receivers[_b0 : _b0 + MAT_BATCH]
    _bm  = calib_rssi_meas[_b0 : _b0 + MAT_BATCH]
    for nm in list(scene.receivers.keys()):
        scene.remove(nm)
    for rx in _brx:
        scene.add(rx)
    try:
        _tp = scene.trace_paths(**_tr_cfg)   # returns 8-tuple
        _traced_list.append((_tp, list(_brx)))
        _meas_list.append(_bm)
    except Exception as e:
        print(f'  batch {_b0//MAT_BATCH+1}: trace_paths failed ({e})')
        continue
    if (_b0 // MAT_BATCH) % 5 == 0:
        print(f'  batch {_b0//MAT_BATCH+1}/{int(np.ceil(len(calib_receivers)/MAT_BATCH))} done')
print(f'Traced {len(_traced_list)} batches OK')

# ── 3. Apply variable values to scene materials ───────────────────────────
def _apply11():
    for mn, vd in mat11_vars.items():
        mat = scene.radio_materials.get(mn)
        if mat is None: continue
        eps_v = tf.clip_by_value(vd['eps'],      1.0, 100.0)
        sig_v = tf.exp(tf.clip_by_value(vd['log_sig'],
                       float(np.log(1e-6)), float(np.log(1e7))))
        s_v   = tf.clip_by_value(vd['s'],        0.0, 1.0)
        try:
            mat.relative_permittivity = eps_v
            mat.conductivity          = sig_v
        except: pass
        for _a in ('scattering_coefficient', 'scattering_coeff'):
            if hasattr(mat, _a):
                try: setattr(mat, _a, s_v); break
                except: pass

# ── 4. Evaluate all batches ────────────────────────────────────────────────
def _eval_all():
    _rs_all, _rm_all = [], []
    for (_tp, _brx), _bm in zip(_traced_list, _meas_list):
        # Restore this batch's receivers before compute_fields()
        for nm in list(scene.receivers.keys()):
            scene.remove(nm)
        for rx in _brx:
            scene.add(rx)
        try:
            _flds = scene.compute_fields(*_tp)
            # paths_to_rssi assumes a.shape[0]=n_rx; verify and fix if transposed
            _a = _flds.a
            if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
            _a = tf.cast(_a, tf.complex64)
            _pwr = tf.abs(_a)**2
            # shape can be (n_rx,n_tx,...) or (n_tx,n_rx,...) depending on API
            # use the axis that matches batch size
            _n_batch = len(_brx)
            if _pwr.shape[0] != _n_batch and _pwr.shape[1] == _n_batch:
                _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
            _nr = tf.shape(_pwr)[0]
            _p  = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
            _rssi = 10.0*tf.experimental.numpy.log10(_p+1e-30)+30.0+RX_EXTRA_GAIN_DB
            _rssi = tf.reshape(_rssi, [-1])
        except Exception as e:
            print(f'  _eval_all batch failed: {e}')
            continue
        _n  = min(len(_rssi), len(_bm))
        _rs = _rssi[:_n]
        _rm = tf.cast(_bm[:_n], tf.float32)
        _vm = tf.math.is_finite(_rs) & (_rs > -150.0)
        if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
            continue
        _rs_all.append(tf.boolean_mask(_rs, _vm))
        _rm_all.append(tf.boolean_mask(_rm, _vm))
    if not _rs_all:
        return None, None
    return tf.concat(_rs_all, 0), tf.concat(_rm_all, 0)

# ── 5. Baseline RMSE ──────────────────────────────────────────────────────
_apply11()
_rs0, _rm0 = _eval_all()
if _rs0 is not None:
    _rmse11_init = float(tf.sqrt(tf.reduce_mean((_rs0 - _rm0)**2)).numpy())
    _mae11_init  = float(tf.reduce_mean(tf.abs(_rs0 - _rm0)).numpy())
    print(f'\nBefore material calib : RMSE={_rmse11_init:.2f} dB  MAE={_mae11_init:.2f} dB  N={len(_rs0)}')
else:
    _rmse11_init = 999.0
    print('ERROR: no valid pairs — check scene/receivers')

# ── 6. Training loop ──────────────────────────────────────────────────────
_all11_vars = []
for vd in mat11_vars.values():
    _all11_vars += [vd['eps'], vd['log_sig'], vd['s']]
_mat11_opt = tf.keras.optimizers.Adam(learning_rate=MAT_LR)

print(f'\nTraining material params ({MAT_STEPS} steps, LR={MAT_LR})')
print('-' * 60)
t0 = time.time()
mat11_hist = {'step': [], 'loss': [], 'rmse': []}

for step in range(MAT_STEPS):
    _step_rs, _step_rm = [], []
    _step_loss = tf.constant(0.0)
    _n_ok = 0
    with tf.GradientTape() as tape:
        _apply11()
        for (_tp, _brx), _bm in zip(_traced_list, _meas_list):
            for nm in list(scene.receivers.keys()):
                scene.remove(nm)
            for rx in _brx:
                scene.add(rx)
            try:
                _flds = scene.compute_fields(*_tp)
                _a = _flds.a
                if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
                _a = tf.cast(_a, tf.complex64)
                _pwr = tf.abs(_a)**2
                _n_batch = len(_brx)
                if _pwr.shape[0] != _n_batch and _pwr.shape[1] == _n_batch:
                    _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
                _nr = tf.shape(_pwr)[0]
                _p  = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
                _rssi = 10.0*tf.experimental.numpy.log10(_p+1e-30)+30.0+RX_EXTRA_GAIN_DB
                _rssi = tf.reshape(_rssi, [-1])
            except Exception:
                continue
            _n  = min(len(_rssi), len(_bm))
            _rs = _rssi[:_n]
            _rm = tf.cast(_bm[:_n], tf.float32)
            _vm = tf.math.is_finite(_rs) & (_rs > -150.0)
            if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
                continue
            _rs_v = tf.boolean_mask(_rs, _vm)
            _rm_v = tf.boolean_mask(_rm, _vm)
            _step_loss = _step_loss + smape_power_loss(_rs_v, _rm_v)
            _step_rs.append(_rs_v)
            _step_rm.append(_rm_v)
            _n_ok += 1
        if _n_ok > 0:
            _step_loss = _step_loss / float(_n_ok)
    if _n_ok == 0:
        print(f'  step {step:4d}: no valid batches — stopping'); break
    _grads = tape.gradient(_step_loss, _all11_vars)
    _mat11_opt.apply_gradients(zip(_grads, _all11_vars))
    lv = float(_step_loss.numpy()) * 100
    mat11_hist['step'].append(step)
    mat11_hist['loss'].append(lv)
    if step % 20 == 0 or step == MAT_STEPS - 1:
        _rmse_v = float(tf.sqrt(tf.reduce_mean(
            (tf.concat(_step_rs,0) - tf.concat(_step_rm,0))**2)).numpy())
        _nz = sum(1 for g in _grads if g is None or float(tf.reduce_sum(tf.abs(g)))==0)
        mat11_hist['rmse'].append(_rmse_v)
        print(f'  step {step:4d}  RMSE={_rmse_v:5.2f} dB  SMAPE={lv:+7.2f}  zero_grads={_nz}/{len(_all11_vars)}  t={time.time()-t0:.0f}s')

print('-' * 60)
print(f'Done in {time.time()-t0:.1f}s')

# ── 7. Final RMSE ─────────────────────────────────────────────────────────
_apply11()
_rs_f, _rm_f = _eval_all()
if _rs_f is not None:
    _rmse11_final = float(tf.sqrt(tf.reduce_mean((_rs_f - _rm_f)**2)).numpy())
    _mae11_final  = float(tf.reduce_mean(tf.abs(_rs_f - _rm_f)).numpy())
    print(f'\nAfter  material calib : RMSE={_rmse11_final:.2f} dB  MAE={_mae11_final:.2f} dB')
    print(f'RMSE improvement      : {_rmse11_init - _rmse11_final:+.2f} dB')

# ── 8. Calibrated parameter table ─────────────────────────────────────────
print(f'\n{"Material":<30} {"Param":<8} {"Init":>10} {"Final":>12} {"Delta":>8}')
print('-' * 72)
for mn, vd in mat11_vars.items():
    iv = mat11_init[mn]
    eps_c = float(tf.clip_by_value(vd['eps'], 1.0, 100.0).numpy())
    sig_c = float(tf.exp(tf.clip_by_value(vd['log_sig'],
                  float(np.log(1e-6)), float(np.log(1e7)))).numpy())
    s_c   = float(tf.clip_by_value(vd['s'], 0.0, 1.0).numpy())
    print(f'  {mn:<28}  eps_r  {iv["eps"]:>10.3f}  {eps_c:>12.3f}  {eps_c-iv["eps"]:>+8.3f}')
    print(f'  {"":28}  sigma  {iv["sig"]:>10.5f}  {sig_c:>12.5f}  {sig_c-iv["sig"]:>+8.5f}')
    print(f'  {"":28}  S      {iv["s"]:>10.4f}  {s_c:>12.4f}  {s_c-iv["s"]:>+8.4f}')


In [ ]:
n_mats   = len(trainable_mats)
colors   = plt.cm.tab10(np.linspace(0, 1, max(n_mats, 1)))
mat_list = list(trainable_mats.keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['step'], history['loss_db'], 'k-', lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss (dB)')
axes[0].set_title('Calibration Loss'); axes[0].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[1].plot(history['step'], history['eps_r'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[1].set_xlabel('Step'); axes[1].set_ylabel('ε_r')
axes[1].set_title('Relative Permittivity'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.4)

for i, mn in enumerate(mat_list):
    axes[2].plot(history['step'], history['log_sigma'][mn],
                 label=mn.replace('_train',''), color=colors[i])
axes[2].set_xlabel('Step'); axes[2].set_ylabel('log σ (S/m)')
axes[2].set_title('Conductivity (log scale)'); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration_convergence.png'), dpi=150)
plt.show()

In [ ]:
print(f'{"Material":<30} {"Param":<8} {"Initial":>10} {"Calibrated":>12} {"Delta":>8}')
print('=' * 75)

calib_results = {}
for mn, mat in trainable_mats.items():
    orig = orig_params.get(mn, {})
    row  = {}
    for pname, attr_candidates, key in [
        ('eps_r', ['relative_permittivity'], 'eps_r'),
        ('sigma', ['conductivity'],           'sigma'),
        ('S',     ['scattering_coefficient', 'scattering_coeff'], 'S'),
    ]:
        init_val = orig.get(key, float('nan'))
        cal_val  = float('nan')
        for a_ in attr_candidates:
            if hasattr(mat, a_):
                try: cal_val = _safe(getattr(mat, a_)); break
                except: pass
        delta = cal_val - init_val if not np.isnan(init_val) else float('nan')
        print(f'  {mn.replace(_train_suffix,""):<28} {pname:<8} '
              f'{init_val:>10.4f} {cal_val:>12.4f} {delta:>+8.4f}')
        row[pname] = {'initial': init_val, 'calibrated': cal_val}
    calib_results[mn] = row

out_json = os.path.join(OUTPUT_DIR, 'calibration_results.json')
with open(out_json, 'w') as f:
    json.dump(calib_results, f, indent=2)
print(f'\nCalibration JSON saved to {out_json}')

---
## CELL 12 · TX Orientation Optimization

Optimises the transmitter antenna pointing direction (`tx.orientation`) to maximise coverage.

| Parameter | Value |
|-----------|-------|
| Optimizer | RMSprop |
| Loss | −E[log₂(1 + SNR × path_gain)] |
| Variable | `tx.orientation` (azimuth, tilt) as `tf.Variable` |
| Steps | `ORI_STEPS` |

Run after Cell 11b (material calibration) for best results.


In [ ]:
tx_name = list(scene.transmitters.keys())[0]
tx      = scene.transmitters[tx_name]

_ori_init = [0.0, 0.0, 0.0]
try: _ori_init = [_safe(tx.orientation[i]) for i in range(3)]
except: pass

tx.orientation = tf.Variable(_ori_init, dtype=tf.float32, name='tx_orientation')
print(f'TX "{tx_name}"  orientation = {_ori_init}  → tf.Variable')

def cm_capacity_loss(sc, cell_size=10.0, n_samp=ORI_NUM_SAMP):
    """Loss = −E[log₂(1 + SNR_scale × path_gain)]  (diff-rt Learning_Orientation)."""
    try:
        cm = sc.coverage_map(
            cm_cell_size        = cell_size,
            max_depth           = 3,
            num_samples         = n_samp,
            los                 = True,
            specular_reflection = True,
            diffuse_reflection  = False,
            refraction          = False,
            diffraction         = True,
        )
        pg = None
        for attr in ('path_gain', 'as_tensor'):
            if hasattr(cm, attr):
                val = getattr(cm, attr)
                pg  = val() if callable(val) else val
                break
        if pg is None: raise AttributeError('no path_gain')
        pg_flat  = tf.reshape(tf.cast(pg[0], tf.float32), [-1])
        capacity = tf.reduce_mean(
            tf.math.log(1.0 + SNR_SCALE * pg_flat) / tf.math.log(2.0))
        return -capacity, cm
    except Exception as e:
        print(f'  cm_capacity_loss error: {e}')
        return tf.constant(0.0), None

print(f'SNR_SCALE = {SNR_SCALE:.2e}')

In [ ]:
ori_optimizer = tf.keras.optimizers.RMSprop(learning_rate=ORI_LR)
ori_history   = {'step': [], 'rate_bit': [], 'orientation': []}

print(f'TX orientation optimization – {ORI_STEPS} steps  RMSprop LR={ORI_LR}')
print('-' * 60)

cm_before_np = None
t0 = time.time()
for step in range(ORI_STEPS):
    with tf.GradientTape() as tape:
        loss_val, cm_opt = cm_capacity_loss(scene, cell_size=10.0, n_samp=ORI_NUM_SAMP)

    if step == 0 and cm_opt is not None:
        cm_before_np = _cm_to_numpy(cm_opt)

    grads    = tape.gradient(loss_val, tape.watched_variables())
    valid_gv = [(g, v) for g, v in zip(grads, tape.watched_variables()) if g is not None]
    if valid_gv: ori_optimizer.apply_gradients(valid_gv)

    rate = float(-loss_val.numpy())
    ori  = list(tx.orientation.numpy())
    ori_history['step'].append(step)
    ori_history['rate_bit'].append(rate)
    ori_history['orientation'].append(ori)
    print(f'  step {step:3d}  rate={rate:.4f} bit  '
          f'ori=[{", ".join(f"{o:.3f}" for o in ori)}]  t={time.time()-t0:.0f}s', end='\r')

print()
print('-' * 60)
ori_final = list(tx.orientation.numpy())
print(f'Initial orientation  : {_ori_init}')
print(f'Optimized orientation: {[round(o, 4) for o in ori_final]}')
rate_initial = ori_history['rate_bit'][0]  if ori_history['rate_bit'] else 0
rate_final   = ori_history['rate_bit'][-1] if ori_history['rate_bit'] else 0
print(f'Rate improvement     : {rate_initial:.4f} → {rate_final:.4f} bit  (+{rate_final-rate_initial:.4f})')

---
## CELL 14 · CNN+MLP Path Loss Predictor — Sionna RT Hybrid Model

Combines physics-based Sionna RT simulation output with a data-driven CNN  
to correct residual errors that ray tracing cannot model (vegetation attenuation,  
diffraction edge effects, near-field clutter).

**Architecture:**

```
nDSM patch (64×64) ──► Conv2D(32,3) ──► Conv2D(64,3) ──► GlobalAvgPool ──► FC(64) ──┐
                                                                                       ├──► FC(64) ──► RSSI_pred
[rssi_sim, dist_km,  ──────────────────────────────────────────────────── FC(32) ──┘
 tx_h, rx_h, az_deg]
```

**Inputs:**

| Input | Shape | Source |
|-------|-------|--------|
| nDSM patch | (64, 64, 1) | `ndsm.tif` cropped 64m×64m centred on RX |
| `rssi_sim` | scalar | `rssi_sim_cached` from Cell 10b pre-trace |
| `dist_km` | scalar | TX–RX distance from receiver_locations.csv |
| `tx_h_m` | scalar | TX height AGL |
| `rx_h_m` | scalar | RX height AGL |
| `az_deg` | scalar | TX→RX azimuth bearing (0–360°) |

**Output:** Predicted RSSI (dBm) — trained against measured RSSI  
**Loss:** MSE on dBm  
**Split:** 80% train / 20% test (random seed fixed)


In [ ]:
# ====================================================================
# CELL 14 — CNN+MLP Hybrid: Sionna RT output + nDSM patch
# Inputs : nDSM 64×64 patch + [rssi_sim, dist, tx_h, rx_h, azimuth]
# Output : predicted RSSI (dBm)
# Labels : measured RSSI from Ofcom CSV
# ====================================================================
import os, time
import numpy as np
import tensorflow as tf
import rasterio
from rasterio.windows import Window

# ── Config ────────────────────────────────────────────────────────────
PATCH_M      = 64          # nDSM patch side length in metres (= pixels at 1m res)
PATCH_HALF   = PATCH_M // 2
CNN_EPOCHS   = 100
CNN_LR       = 1e-3
CNN_BATCH    = 32
TRAIN_SPLIT  = 0.8
RANDOM_SEED  = 42
NDSM_PATH    = os.path.join(BASE_DIR, 'ndsm.tif')

print('=' * 70)
print('CELL 14 — CNN+MLP Hybrid Path Loss Predictor')
print('=' * 70)

# ── 1. Load nDSM and build patch extractor ────────────────────────────
print(f'Loading nDSM from {NDSM_PATH} ...')
_ndsm_src  = rasterio.open(NDSM_PATH)
_ndsm_data = _ndsm_src.read(1).astype(np.float32)   # (H, W)
_ndsm_data = np.clip(_ndsm_data, 0.0, 120.0)        # cap at 120m (mast outlier)
_ndsm_max  = 120.0
print(f'  nDSM shape  : {_ndsm_data.shape}')
print(f'  CRS         : {_ndsm_src.crs}')
print(f'  Transform   : {_ndsm_src.transform}')

def _get_patch(local_x, local_y):
    """Extract 64×64 nDSM patch centred on local XY (scene coords → BNG)."""
    # Convert scene local XY back to BNG
    bng_e = local_x + scene_origin_x   # scene_origin_x from Cell 3
    bng_n = local_y + scene_origin_y
    # rasterio row/col
    row, col = _ndsm_src.index(bng_e, bng_n)
    r0 = max(0, row - PATCH_HALF)
    c0 = max(0, col - PATCH_HALF)
    r1 = r0 + PATCH_M
    c1 = c0 + PATCH_M
    # guard against edge overflow
    H, W = _ndsm_data.shape
    r1 = min(r1, H); r0 = r1 - PATCH_M
    c1 = min(c1, W); c0 = c1 - PATCH_M
    r0 = max(0, r0); c0 = max(0, c0)
    patch = _ndsm_data[r0:r1, c0:c1]
    if patch.shape != (PATCH_M, PATCH_M):
        patch = np.pad(patch,
                       ((0, PATCH_M - patch.shape[0]),
                        (0, PATCH_M - patch.shape[1])),
                       mode='constant', constant_values=0.0)
    return (patch / _ndsm_max).astype(np.float32)   # normalise [0,1]

# ── 2. Build dataset from calib_receivers + rssi_sim_cached ───────────
# rssi_sim_cached must exist from Cell 10b (scalar offset pre-trace)
# If Cell 10b was skipped, run it first or this cell will raise an error
if 'rssi_sim_cached' not in dir():
    raise RuntimeError('rssi_sim_cached not found — run Cell 10b first')

print(f'\nBuilding dataset ({len(calib_receivers)} receivers) ...')

# TX position for distance/azimuth computation
_tx      = list(scene.transmitters.values())[0]
_tx_pos  = _tx.position.numpy()   # [x, y, z] local
_tx_h    = float(_tx_pos[2])

patches    = []
scalars    = []
labels     = []
valid_idx  = []

for i, (rx, rssi_sim_v, rssi_meas_v) in enumerate(
        zip(calib_receivers,
            rssi_sim_cached.numpy(),
            calib_rssi_meas.numpy())):

    # Skip receivers with no valid simulation path
    if not np.isfinite(rssi_sim_v) or rssi_sim_v < -150.0:
        continue
    if not np.isfinite(rssi_meas_v):
        continue

    rx_pos = rx.position.numpy()
    dx     = rx_pos[0] - _tx_pos[0]
    dy     = rx_pos[1] - _tx_pos[1]
    dist_m = float(np.sqrt(dx**2 + dy**2))
    dist_km = dist_m / 1000.0
    rx_h    = float(rx_pos[2])
    az_deg  = float(np.degrees(np.arctan2(dx, dy)) % 360.0)

    # nDSM patch centred on receiver
    patch = _get_patch(float(rx_pos[0]), float(rx_pos[1]))

    # Scalar feature vector — all normalised
    sc = np.array([
        rssi_sim_v / -150.0,       # sim RSSI normalised (−150→0, 0→1 scale inverted)
        dist_km    / 10.0,         # distance km / 10
        _tx_h      / 100.0,        # TX height / 100m
        rx_h       / 50.0,         # RX height / 50m
        az_deg     / 360.0,        # azimuth fraction
    ], dtype=np.float32)

    patches.append(patch[..., np.newaxis])   # (64,64,1)
    scalars.append(sc)                        # (5,)
    labels.append(rssi_meas_v)               # scalar dBm
    valid_idx.append(i)

N = len(patches)
print(f'Dataset size : {N} valid receiver pairs')

patches_arr = np.stack(patches, axis=0)   # (N,64,64,1)
scalars_arr = np.stack(scalars, axis=0)   # (N,5)
labels_arr  = np.array(labels, dtype=np.float32)  # (N,)

# ── 3. Train/test split ────────────────────────────────────────────────
rng      = np.random.default_rng(RANDOM_SEED)
idx_all  = np.arange(N)
rng.shuffle(idx_all)
n_train  = int(N * TRAIN_SPLIT)
idx_tr   = idx_all[:n_train]
idx_te   = idx_all[n_train:]

X_patch_tr = patches_arr[idx_tr];  X_patch_te = patches_arr[idx_te]
X_sc_tr    = scalars_arr[idx_tr];  X_sc_te    = scalars_arr[idx_te]
y_tr       = labels_arr[idx_tr];   y_te       = labels_arr[idx_te]
print(f'Train : {len(idx_tr)}   Test : {len(idx_te)}')

# ── 4. Model definition ────────────────────────────────────────────────
def build_cnn_mlp(patch_shape=(PATCH_M, PATCH_M, 1), n_scalar=5):
    # CNN branch — processes nDSM patch
    patch_in = tf.keras.Input(shape=patch_shape, name='ndsm_patch')
    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(patch_in)
    x = tf.keras.layers.MaxPool2D(2)(x)
    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPool2D(2)(x)
    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)

    # MLP branch — processes scalar features
    sc_in = tf.keras.Input(shape=(n_scalar,), name='scalar_feats')
    s = tf.keras.layers.Dense(32, activation='relu')(sc_in)
    s = tf.keras.layers.Dense(32, activation='relu')(s)

    # Merge and predict
    merged = tf.keras.layers.Concatenate()([x, s])
    out    = tf.keras.layers.Dense(64, activation='relu')(merged)
    out    = tf.keras.layers.Dropout(0.2)(out)
    out    = tf.keras.layers.Dense(1, activation='linear', name='rssi_pred')(out)

    model = tf.keras.Model(inputs=[patch_in, sc_in], outputs=out)
    return model

model_cnn = build_cnn_mlp()
model_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(CNN_LR),
    loss='mse',
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)
model_cnn.summary()

# ── 5. Training ────────────────────────────────────────────────────────
print(f'\nTraining CNN+MLP ({CNN_EPOCHS} epochs, batch={CNN_BATCH}) ...')
t0 = time.time()

cnn_history = model_cnn.fit(
    x=[X_patch_tr, X_sc_tr],
    y=y_tr,
    validation_data=([X_patch_te, X_sc_te], y_te),
    epochs=CNN_EPOCHS,
    batch_size=CNN_BATCH,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_rmse', patience=15, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5)
    ],
    verbose=0
)

# Print epoch summary every 10 epochs
for ep in range(0, len(cnn_history.history['rmse']), 10):
    tr_rmse  = cnn_history.history['rmse'][ep]
    val_rmse = cnn_history.history['val_rmse'][ep]
    print(f'  epoch {ep+1:3d}  train_RMSE={tr_rmse:.2f} dB  val_RMSE={val_rmse:.2f} dB')
print(f'Done in {time.time()-t0:.1f}s')

# ── 6. Evaluation ──────────────────────────────────────────────────────
y_pred_te = model_cnn.predict([X_patch_te, X_sc_te], verbose=0).flatten()
_rmse_cnn = float(np.sqrt(np.mean((y_pred_te - y_te)**2)))
_mae_cnn  = float(np.mean(np.abs(y_pred_te - y_te)))
_bias_cnn = float(np.mean(y_pred_te - y_te))

# RT-only baseline on same test set
_rssi_sim_te = rssi_sim_cached.numpy()[np.array(valid_idx)[idx_te]]
_rmse_rt     = float(np.sqrt(np.mean((_rssi_sim_te - y_te)**2)))

print(f'\n{"Model":<25} {"RMSE":>8} {"MAE":>8} {"Bias":>8} {"N":>6}')
print('-' * 60)
print(f'  RT only (Cell 10b)    {_rmse_rt:>8.2f} dB  {"—":>8}  {"—":>8}  {len(y_te):>6}')
print(f'  CNN+MLP (this cell)   {_rmse_cnn:>8.2f} dB  {_mae_cnn:>8.2f}  {_bias_cnn:>+8.2f}  {len(y_te):>6}')
print(f'  RMSE improvement      {_rmse_rt - _rmse_cnn:>+8.2f} dB')


---
## CELL 13 · Post-Calibration Analysis

Computes final coverage map and per-receiver error statistics using calibrated material properties.

**Outputs:**
- Final coverage map (calibrated vs pre-calibration)
- Per-receiver: RSSI_sim, RSSI_meas, error (dB), distance (m)
- Summary: RMSE, MAE, Bias, R² across all 1 200 receivers
- CSV export: `simulation_results_calibrated.csv`


In [ ]:
print('Post-calibration path computation ...')
print(f'  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

paths_cal = scene.compute_paths(
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_PS,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = True,
)
print('Done.')

a_np   = _to_numpy(paths_cal.a)
tau_np = _to_numpy(paths_cal.tau)
if a_np.ndim == 6: a_np = a_np[0]

n_rx    = a_np.shape[0]
power   = np.sum(np.abs(a_np)**2, axis=tuple(range(1, a_np.ndim)))
pg_cal  = power
pg_cal_db = 10 * np.log10(pg_cal + 1e-30)

print(f'Post-calibration path gain at {n_rx} receivers:')
print(f'  mean={pg_cal_db.mean():.1f} dB  min={pg_cal_db.min():.1f} dB  max={pg_cal_db.max():.1f} dB')

In [ ]:
records = []
for i, rx in enumerate(receivers[:n_rx]):
    x   = _safe(rx.position[0])
    y   = _safe(rx.position[1])
    z   = _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    pg_pre  = float(pg_at_rx_pre[i]) if i < len(pg_at_rx_pre) else float('nan')
    pg_post = float(pg_cal_db[i])    if i < len(pg_cal_db)    else float('nan')
    records.append({
        'receiver'    : rx.name,
        'lon'         : round(lon, 6),
        'lat'         : round(lat, 6),
        'x_m'         : round(x,   2),
        'y_m'         : round(y,   2),
        'z_m'         : round(z,   3),
        'pg_pre_db'   : round(pg_pre,  2),
        'pg_post_db'  : round(pg_post, 2),
        'delta_pg_db' : round(pg_post - pg_pre, 2) if not np.isnan(pg_pre) else float('nan'),
    })

df_out = pd.DataFrame(records)
out_csv = os.path.join(OUTPUT_DIR, 'receiver_results_calibrated.csv')
df_out.to_csv(out_csv, index=False)
print(f'Saved {len(df_out)} receivers to {out_csv}')
print(df_out.head(10).to_string(index=False))

In [ ]:
print('Computing final calibrated coverage map ...')
cm_final    = scene.coverage_map(
    cm_cell_size        = GRID_SIZE_M,
    max_depth           = MAX_DEPTH,
    num_samples         = NUM_SAMPLES_CM,
    los                 = True,
    specular_reflection = True,
    diffuse_reflection  = True,
    refraction          = True,
    diffraction         = True,
)
cm_final_np    = _cm_to_numpy(cm_final)
pg_pre_db_2d   = 10 * np.log10(cm_pre_np[0]   + 1e-30)
pg_final_db_2d = 10 * np.log10(cm_final_np[0] + 1e-30)

vmin = min(np.nanpercentile(pg_pre_db_2d, 5),  np.nanpercentile(pg_final_db_2d, 5))
vmax = max(np.nanpercentile(pg_pre_db_2d, 99), np.nanpercentile(pg_final_db_2d, 99))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, data, title in [
    (axes[0], pg_pre_db_2d,   'Before Calibration (ITU defaults)'),
    (axes[1], pg_final_db_2d, 'After Calibration (diff-rt)'),
    (axes[2], pg_final_db_2d - pg_pre_db_2d, 'Δ Path Gain (After − Before)'),
]:
    if 'Δ' in title:
        im = ax.imshow(data, origin='lower', cmap='RdYlGn', vmin=-10, vmax=10)
    else:
        im = ax.imshow(data, origin='lower', cmap='jet', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Path Gain (dB)')
    ax.set_title(title); ax.set_xlabel('X cells'); ax.set_ylabel('Y cells')

plt.suptitle('Coverage Map: Before vs After Differentiable RT Calibration', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nAll results saved to:', OUTPUT_DIR)